# Lending Club — Live Demo (works on T4 Colab)

**UE24CS352A ML Mini-Project — Team A-15**  ·  Aman Priyansh and Addanki Harsha Rajeswari

This notebook is for the **review demo**. It is memory-light so it fits in free Colab (T4 or even CPU).

What it does:
1. Downloads the Kaggle dataset.
2. Builds a small, honest training table (2012-2014 loans, leaky columns dropped, ~150k loans).
3. Trains the tuned XGBoost from `02_Improvements.ipynb` (test ROC-AUC ≈ 0.74).
4. Saves the model + preprocessing to `demo_artifacts/` so the Streamlit app (`demo_app.py`) can run offline.
5. Prints 10 **pre-tested** loan numbers that give clean HIGH-risk and LOW-risk results.
6. Launches **Gradio** with a public share link — paste that link anywhere during the review.

**How to run:** Runtime → Change runtime type → **T4 GPU** → Runtime → Run all.
Total time: about 4 minutes (data download is the slow bit).


## 1. Download the data

In [ ]:
# CELL 1 - Download the Kaggle dataset (about 1.6 GB)
# This is the only slow step. It takes ~1 minute on a good connection.
import kagglehub
data_dir = kagglehub.dataset_download("wordsforthewise/lending-club")
print("Data folder:", data_dir)


## 2. Load only what we need, keep RAM low
Why T4 crashed before: the full baseline builds a 550k × 1099 one-hot matrix (~4 GB), plus intermediate copies.
Here we drop the leaky columns, sample 150k training loans, and skip the 900-value `zip_code` one-hot.

In [ ]:
# CELL 2 - Load only what we need, in small pieces (memory-friendly for T4 Colab)
# For the demo we only need:
#   - a modest TRAIN set from 2012-2014 loans
#   - every 2015 loan for the demo picker and for the test
# We DROP the leaky columns at load time (data leakage: credit score after the loan ended).
# We KEEP a small, meaningful set of features so the one-hot explosion fits in RAM.
import pandas as pd, numpy as np, gc

LEAK = ['last_fico_range_high', 'last_fico_range_low', 'out_prncp', 'out_prncp_inv']

# Features kept for the honest model. Chosen because they are:
#   - available when the loan is applied for (not after it ends)
#   - the ones XGBoost used most in our full S3 run (feature importance)
#   - compact (addr_state has 51 values; we drop the 900-value zip_code to save RAM)
NUM = ['loan_amnt', 'funded_amnt', 'int_rate', 'installment', 'annual_inc', 'dti',
       'fico_range_low', 'fico_range_high', 'delinq_2yrs', 'inq_last_6mths',
       'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc',
       'mort_acc', 'tot_cur_bal', 'avg_cur_bal', 'bc_util', 'pct_tl_nvr_dlq',
       'num_actv_rev_tl', 'num_bc_sats', 'num_rev_accts', 'num_sats',
       'acc_open_past_24mths', 'total_bc_limit']
CAT = ['term', 'grade', 'sub_grade', 'home_ownership', 'verification_status',
       'purpose', 'addr_state', 'emp_length', 'application_type']
KEEP_COLS = NUM + CAT + ['loan_status', 'issue_d']

csv_file = f"{data_dir}/accepted_2007_to_2018q4.csv/accepted_2007_to_2018Q4.csv"
raw = pd.read_csv(csv_file, usecols=KEEP_COLS, low_memory=False)
print("Full file:", raw.shape)

# Keep finished loans issued 2012-2015 only (same filter as the paper and the baseline notebook)
raw['issue_year'] = pd.to_datetime(raw['issue_d'], format='%b-%Y').dt.year
clean = raw[raw['issue_year'].between(2012, 2015) &
            raw['loan_status'].isin(['Fully Paid', 'Charged Off', 'Default'])].copy()
del raw; gc.collect()

# Honest split: train on 2012-2014, test on every 2015 loan (what a real lender would do)
train_full = clean[clean['issue_year'] <= 2014].copy()
test_full  = clean[clean['issue_year'] == 2015].copy()
print(f"Train pool: {len(train_full):,} loans (2012-2014)")
print(f"Test:       {len(test_full):,} loans (2015)   <-- used by the demo picker")

# Sample 150,000 training loans. Enough signal for ROC-AUC within ~0.01 of the full run,
# while keeping the one-hot table under 4 GB on a 12 GB machine.
train = train_full.sample(n=150_000, random_state=0).reset_index(drop=True)
test  = test_full.reset_index(drop=True)
del clean, train_full, test_full; gc.collect()
print(f"Using TRAIN: {len(train):,}  TEST: {len(test):,}")


## 3. Preprocess (fills, one-hot, standardise)

In [ ]:
# CELL 3 - Build the matrix: parse percent-strings, fill empties, one-hot categoricals, standardise
# Memory-light: everything in float32, standardise in place.
def parse_pct(s):
    return pd.to_numeric(s.astype(str).str.replace('%', '', regex=False), errors='coerce') / 100.0

def prep(df):
    df = df.copy()
    df['int_rate']   = parse_pct(df['int_rate'])
    df['revol_util'] = parse_pct(df['revol_util'])
    # Simple fills: median for numeric, "missing" for categoricals
    for c in NUM:
        df[c] = pd.to_numeric(df[c], errors='coerce')
    return df

train_p = prep(train); test_p = prep(test)
med = train_p[NUM].median()           # medians from TRAIN only (no leak from test)
for c in NUM:
    train_p[c] = train_p[c].fillna(med[c])
    test_p[c]  = test_p[c].fillna(med[c])
for c in CAT:
    train_p[c] = train_p[c].fillna('missing').astype(str)
    test_p[c]  = test_p[c].fillna('missing').astype(str)

# One-hot the categoricals with pandas; combine so train and test have the same columns.
cats = {c: sorted(train_p[c].unique()) for c in CAT}
names = NUM + [f"{c}_{v}" for c in CAT for v in cats[c]]
print(f"Features after one-hot: {len(names)}")

def to_matrix(df):
    X = np.zeros((len(df), len(names)), dtype=np.float32)
    X[:, :len(NUM)] = df[NUM].to_numpy(dtype=np.float32)
    j = len(NUM)
    for c in CAT:
        codes = pd.Categorical(df[c], categories=cats[c]).codes.astype(np.int64)
        rows = np.where(codes >= 0)[0]
        X[rows, j + codes[rows]] = 1.0
        j += len(cats[c])
    np.nan_to_num(X, copy=False)
    return X

Xtr = to_matrix(train_p)
Xte = to_matrix(test_p)

# Standardise using TRAIN means / stds only (fit on train, applied to test)
mean = Xtr.mean(0, dtype=np.float64).astype(np.float32)
std  = Xtr.std(0,  dtype=np.float64).astype(np.float32); std[std == 0] = 1.0
Xtr -= mean; Xtr /= std
Xte -= mean; Xte /= std

# Labels: 1 = Fully Paid, 0 = Default / Charged Off (same mapping as the paper)
ytr = (train_p['loan_status'] == 'Fully Paid').astype(int).to_numpy()
yte = (test_p['loan_status']  == 'Fully Paid').astype(int).to_numpy()
del train_p, test_p; gc.collect()
print(f"Xtr: {Xtr.shape}  Xte: {Xte.shape}  Default share: train {(1-ytr.mean()):.3f}  test {(1-yte.mean()):.3f}")


## 4. Train the tuned XGBoost (honest model, no leakage)

In [ ]:
# CELL 4 - Train tuned XGBoost (the honest model from the improvements notebook)
# Settings came from the random search in 02_Improvements.ipynb (best validation ROC-AUC):
#   depth 6, learning rate 0.03, min_child_weight 5, colsample 0.3, subsample 1.0, reg_lambda 10
# On T4 GPU this trains in ~30 seconds. On CPU, ~2-3 minutes.
import xgboost as xgb, time
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.model_selection import train_test_split

# Hold out 15% of TRAIN as validation (for early stopping)
Xtr2, Xv, ytr2, yv = train_test_split(Xtr, ytr, test_size=0.15, random_state=0, stratify=ytr)

try_gpu = {'tree_method': 'hist', 'device': 'cuda'}      # T4/L4/A100
try_cpu = {'tree_method': 'hist'}                        # CPU fallback

base_params = dict(n_estimators=1500, learning_rate=0.03, max_depth=6, subsample=1.0,
                   colsample_bytree=0.3, min_child_weight=5, reg_lambda=10.0,
                   eval_metric='auc', early_stopping_rounds=50, random_state=0)

def fit(params_backend):
    m = xgb.XGBClassifier(**base_params, **params_backend)
    m.fit(Xtr2, ytr2, sample_weight=compute_sample_weight('balanced', ytr2),
          eval_set=[(Xv, yv)], verbose=False)
    return m

t = time.time()
try:
    model = fit(try_gpu)
    backend = 'GPU'
except Exception as e:
    print(f"GPU not available ({type(e).__name__}), falling back to CPU")
    model = fit(try_cpu)
    backend = 'CPU'
print(f"Trained on {backend} in {time.time()-t:.1f}s  ({model.best_iteration+1} trees kept)")

# Evaluate on the untouched 2015 test set
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score
proba_paid = model.predict_proba(Xte)[:, 1]
proba_default = 1 - proba_paid
print(f"\nTest ROC-AUC: {roc_auc_score(yte, proba_paid):.3f}  (full-data honest run: 0.744)")
print(f"Test PR-AUC (Default): {average_precision_score(1-yte, proba_default):.3f}")


## 5. Save model + preprocessing (for Streamlit)

In [ ]:
# CELL 5 - Save model + preprocessing so the Streamlit demo can be run offline later
import pickle, os
os.makedirs('demo_artifacts', exist_ok=True)
pickle.dump({
    'num_cols': NUM, 'cat_cols': CAT, 'cat_values': cats, 'feature_names': names,
    'median': med.to_dict(), 'mean': mean, 'std': std,
}, open('demo_artifacts/preproc.pkl', 'wb'))
model.save_model('demo_artifacts/xgb_tuned.json')     # XGBoost's own format (small, portable)
print("Saved to demo_artifacts/:  xgb_tuned.json  preproc.pkl")
print("Download the folder and run streamlit locally with these files.")


## 6. Pre-tested loan numbers for the demo
Each row below is a 2015 test loan the model got right. Use these numbers in the demo if you want
a sure-fire story (3 HIGH-risk loans that defaulted; 3 LOW-risk loans that paid off).

In [ ]:
# CELL 6 - Pre-computed test cases that always give clean, interpretable results
# Use these loan numbers in the demo if you want a safe story to tell. Each one is a 2015 loan.
import numpy as np

# Rank the 2015 test loans by predicted default probability
order_by_risk = np.argsort(-proba_default)   # highest-risk first

cases = []
# 3 clearly high-risk loans that actually DEFAULTED (model got it right)
high_defaulted = [int(i) for i in order_by_risk if yte[i] == 0][:5]
# 3 clearly low-risk loans that actually PAID OFF (model got it right)
low_paid = [int(i) for i in order_by_risk[::-1] if yte[i] == 1][:5]

print("GOOD TEST CASES FOR THE DEMO")
print("=" * 72)
print(f"{'#':>6}  {'Row':>6}  {'Risk':>7}  {'Model':>12}  {'Truth':>12}  {'Amount':>8}  Grade")
print("-" * 72)
for lbl, rows in [("HIGH RISK + DEFAULTED", high_defaulted),
                  ("LOW  RISK + PAID OFF ", low_paid)]:
    print(f"\n{lbl}:")
    for i in rows:
        r = test.iloc[i]
        risk = proba_default[i] * 100
        verdict = 'HIGH risk' if risk >= 50 else 'LOW risk'
        truth = 'Defaulted' if yte[i] == 0 else 'Paid off'
        print(f"  {len(cases):>4}  {i:>6}  {risk:>5.1f}%  {verdict:>12}  {truth:>12}  ${int(r['loan_amnt']):>6}  {r['grade']}-{r['sub_grade']}")
        cases.append(i)

# Save for the Streamlit demo
pickle.dump({'rows': [int(i) for i in cases], 'test_full': test}, open('demo_artifacts/test_cases.pkl', 'wb'))
print(f"\nSaved {len(cases)} tested test cases to demo_artifacts/test_cases.pkl")


## 7. LIVE DEMO — Gradio with a public share link
Run this cell during the review. The public URL works on any laptop or phone.

In [ ]:
# CELL 7 - LIVE DEMO (Gradio) - the easiest way to show the model during the review
# Starts a small web page with a PUBLIC share link that works on any laptop or phone.
!pip -q install gradio
import gradio as gr

show_cols = ['loan_amnt', 'term', 'int_rate', 'installment', 'grade', 'sub_grade',
             'annual_inc', 'dti', 'fico_range_low', 'fico_range_high',
             'emp_length', 'home_ownership', 'purpose', 'addr_state']

def explain(loan_number):
    i = int(min(max(loan_number, 0), len(test) - 1))
    r = test.iloc[i]
    # Build a tidy details table
    details = pd.DataFrame({'Feature': show_cols, 'Value': [str(r[c]) for c in show_cols]})
    risk_pct = proba_default[i] * 100
    verdict = 'HIGH risk of default' if risk_pct >= 50 else 'LOW risk of default'
    truth = 'Defaulted / Charged Off' if yte[i] == 0 else 'Fully Paid'
    model_says = f"Predicted chance of default: {risk_pct:.1f}%  →  {verdict}"
    return details, model_says, f"What really happened: {truth}"

tested_cases = ", ".join(str(c) for c in cases)
desc = f"""Tuned XGBoost trained on 2012-2014 loans WITHOUT the leaky credit-score columns, tested on 2015 loans.
Test ROC-AUC: {roc_auc_score(yte, proba_paid):.3f}. Try these pre-tested loan numbers: {tested_cases}"""

demo = gr.Interface(
    fn=explain,
    inputs=gr.Number(value=cases[0], precision=0,
                     label=f'Loan number (0 to {len(test)-1:,}) from the 2015 test set'),
    outputs=[gr.Dataframe(label='Loan application details'),
             gr.Textbox(label='Model says'),
             gr.Textbox(label='Truth (what actually happened)')],
    title='Lending Club default risk — honest model (no data leakage)',
    description=desc,
    examples=[[c] for c in cases])
demo.launch(share=True)   # share=True gives a public URL that works for the review panel
